# AuspexAI evidence loader — analysis from a verified dataset

Every AuspexAI experiment ends in an **evidence bundle**: your results,
the work-unit inputs they were computed from, the COSE receipts, the
Rekor-anchored result-set attestation, and a signed proof-of-transfer.
`load_verified` runs the **entire verification chain** — custody signature,
attestation, root unification, completeness, input binding, per-result
worker signatures — and only then hands you a DataFrame. If any check
fails, it raises instead of loading: your analysis *begins* from a
cryptographically verified dataset.

Needs the analysis extra: `pip install 'auspexai-tenant[analysis]'`


## 1. Take custody of the bundle

From a shell (verifies on download; the file is yours to keep — the
network re-verifies forever but never re-delivers after age-off):

```bash
auspexai-tenant experiment export exp-XXXXXXXX \
    --coordinator https://coord.auspexai.network --key <your-key>
```


In [ ]:
from auspexai_tenant.evidence import load_verified

df = load_verified("exp-XXXXXXXX-bundle.json")
df.head()

Pin the signing keys against the published roster
([`AUTHORIZED_SIGNERS.md`](https://github.com/auspexai/.github/blob/main/security/AUTHORIZED_SIGNERS.md))
so a valid signature also proves *who* signed — and add the online Rekor
transparency-log check when you have network:


In [ ]:
AUTHORIZED = ["<pubkey hex from AUTHORIZED_SIGNERS.md>"]
df = load_verified(
    "exp-XXXXXXXX-bundle.json",
    authorized_signers=AUTHORIZED,
    check_rekor=True,
)

## 2. Analyze

One row per consensus result. Work-unit **inputs** (your parameters)
flatten to `input.*` columns; result **outputs** to `output.*`. Round or
sweep coordinates are your tenant's semantics — derive them from your
input columns or unit-id convention (the vigiles drift tenant encodes
`<run>-<probe>-r<round>`):


In [ ]:
df["round"] = df["unit_id"].str.extract(r"-r(\d+)$").astype(int)
df.groupby("round")["output.response_sha256"].nunique()  # drift = new hashes per round

In [ ]:
df.describe(include="all")

## 3. Hand off to your tools

Parquet/CSV open the Excel, Tableau, and R door. Or skip Python entirely
with the CLI — it verifies first and writes nothing from a bad bundle:

```bash
auspexai-tenant bundle table exp-XXXXXXXX-bundle.json -o results.parquet
auspexai-tenant bundle verify exp-XXXXXXXX-bundle.json   # re-verify forever
```


In [ ]:
df.to_parquet("results.parquet", index=False)